# Attention Head Probing

Probes what information attention heads encode: positional information, token identity, specialization profiles, output norms, and value rank.

**References:**
- Voita et al. (2019) "Analyzing Multi-Head Self-Attention"
- Clark et al. (2019) "What Does BERT Look At?"

## Why This Matters

Attention head probing uses targeted probes to understand what information each head encodes in its output. Positional probes, identity probes, and specialization measurements characterize heads from the perspective of what they compute, complementing pattern-based analysis.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_probing import (
    head_positional_probe,
    head_token_identity_probe,
    head_specialization_profile,
    head_output_norm_analysis,
    head_value_rank_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Positional probe
pos = head_positional_probe(model, tokens)
print(f"Most positional head: {pos['most_positional_head']}")
print(f"Least positional head: {pos['least_positional_head']}")
print(f"\nLocality scores:")
for l in range(cfg.n_layers):
    scores = [f"{pos['locality_score'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")

In [ ]:
# 2. Token identity probe
tid = head_token_identity_probe(model, tokens)
print(f"Most identity-preserving: {tid['most_identity_preserving']}")
print(f"Most transforming: {tid['most_transforming']}")
print(f"\nIdentity scores:")
for l in range(cfg.n_layers):
    scores = [f"{tid['identity_scores'][l,h]:+.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")

In [ ]:
# 3. Specialization profile
spec = head_specialization_profile(model, tokens)
print("Head specialization labels:")
for l in range(cfg.n_layers):
    labels = [spec['specialization_labels'][l,h] for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {labels}")
print(f"\nBOS scores:")
for l in range(cfg.n_layers):
    scores = [f"{spec['bos_scores'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")

In [ ]:
# 4. Output norm analysis
norms = head_output_norm_analysis(model, tokens)
print(f"Largest head: {norms['largest_head']}")
print(f"Smallest head: {norms['smallest_head']}")
print(f"\nOutput norms:")
for l in range(cfg.n_layers):
    vals = [f"{norms['output_norms'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(vals)} (total={norms['layer_total_norms'][l]:.3f})")

In [ ]:
# 5. Value rank analysis
vr = head_value_rank_analysis(model, tokens, top_k=4)
print(f"Most low-rank head: {vr['most_low_rank']}")
print(f"Most full-rank head: {vr['most_full_rank']}")
print(f"Mean effective rank: {vr['mean_rank']:.2f}")
print(f"\nEffective ranks:")
for l in range(cfg.n_layers):
    ranks = [f"{vr['effective_ranks'][l,h]:.2f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(ranks)}")